# Baseline: Speech Enhancement using a Frame-Level MLP
This notebook illustrates a supervised speech enhancement system. We train a Multi-Layer Perceptron (MLP) to predict an Ideal Ratio Mask (IRM).

## Core Concepts
* Temporal Context: We don't just look at one frame of audio; we look at a window of 11 frames to capture the "shape" of speech phonemes.
* The Masking Approach: Instead of generating speech, the model learns a "gain" between 0 and 1 for every frequency bin.
* Log-Magnitude Features: We train on log-scale magnitudes because they better represent how human hearing perceives loudness.

## 1. Environment Setup
We begin by importing the necessary libraries and ensuring our custom evaluation scripts are accessible.

In [7]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display
from pathlib import Path
from tqdm import tqdm

# Visualization helper
def plot_spectrogram(spec, title="Spectrogram"):
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(librosa.amplitude_to_db(spec, ref=np.max), 
                             y_axis='log', x_axis='time', sr=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

## 2. The Dataset and the IRM Target
The Ideal Ratio Mask (IRM) is our training target. It is mathematically defined to provide a balance between noise suppression and speech naturalness. Target Math:$$IRM(t, f) = \sqrt{\frac{|S(t,f)|^2}{|S(t,f)|^2 + |N(t,f)|^2}}$$

In [8]:
class IRMDataset(Dataset):
    """
    Handles on-the-fly IRM calculation and temporal windowing.
    We use 5 frames of context on either side of the target frame.
    """
    def __init__(self, npz_path, context_frames=5, mean=None, std=None):
        self.context = context_frames
        data = np.load(npz_path)
        
        # Load log-mags for input, linear-mags for target IRM calculation
        self.noisy_log_mag = data['noisy_magnitude'].astype(np.float32)
        clean_mag = np.exp(data['clean_magnitude'].astype(np.float32))
        noisy_mag = np.exp(data['noisy_magnitude'].astype(np.float32))
        
        # Calculate Square-Root IRM
        clean_power = clean_mag ** 2
        noise_power = np.maximum(noisy_mag ** 2 - clean_power, 1e-10)
        self.irm = np.sqrt(clean_power / (clean_power + noise_power + 1e-10))
        self.irm = np.clip(self.irm, 0.0, 1.0).astype(np.float32)

        # Flatten indices (chunk, frame)
        n_chunks, n_frames, n_bins = self.irm.shape
        self.indices = [(c, f) for c in range(n_chunks) 
                        for f in range(context_frames, n_frames - context_frames)]
        
        # Normalization (Standardization)
        if mean is None:
            self.mean = self.noisy_log_mag.mean()
            self.std = self.noisy_log_mag.std()
        else:
            self.mean, self.std = mean, std

    def __getitem__(self, idx):
        c, f = self.indices[idx]
        # Concatenate 11 frames (5+1+5) into a single 1D vector
        window = self.noisy_log_mag[c, f-self.context : f+self.context+1, :].flatten()
        window = (window - self.mean) / (self.std + 1e-8)
        target = self.irm[c, f, :]
        return torch.tensor(window), torch.tensor(target)

    def __len__(self):
        return len(self.indices)

## 3. MLP Architecture
The MLP acts as a nonlinear regression engine. It takes a high-dimensional vector representation the noisy context and compresses/expands its hidden layers to find the "mask" pattern.
* Input: $11 \text{ frames} \times 257 \text{ bins} = 2827$ Neurons.
* Hidden Layers : 4 layers of 1024 neurons each.
* Output : 257 neurons (one mask value for each frequency bin).

In [9]:
class SpeechMLP(nn.Module):
    def __init__(self, input_dim=2827, output_dim=257):
        super().__init__()
        self.net = nn.Sequential(
            # Layer 1
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            # Layer 2-4
            nn.Linear(1024, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.1), nn.Dropout(0.2),
            nn.Linear(1024, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.1), nn.Dropout(0.2),
            nn.Linear(1024, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.1), nn.Dropout(0.2),
            # Output
            nn.Linear(1024, output_dim),
            nn.Sigmoid() # Constrains output to [0, 1]
        )

    def forward(self, x):
        return self.net(x)

model = SpeechMLP()
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters()):,}")

Total Trainable Parameters: 6,316,289


$$\begin{array}{c}
\text{{\bf Input Layer}} \\
\underbrace{\circ \circ \circ \dots \dots \dots \circ}_{2827} \\
\text{\small (11 Frames } \times \text{ 257 Bins)} \\
\downarrow \\
\text{{\bf Hidden Layer 1}} \\
\underbrace{\bullet \bullet \bullet \dots \bullet}_{1024} \\
\downarrow \\
\text{{\bf Hidden Layer 2}} \\
\underbrace{\bullet \bullet \bullet \dots \bullet}_{1024} \\
\downarrow \\
\text{{\bf Hidden Layer 3}} \\
\underbrace{\bullet \bullet \bullet \dots \bullet}_{1024} \\
\downarrow \\
\text{{\bf Hidden Layer 4}} \\
\underbrace{\bullet \bullet \bullet \dots \bullet}_{1024} \\
\downarrow \\
\text{{\bf Output Layer}} \\
\underbrace{\diamond \diamond \dots \diamond}_{257} \\
\text{\small (Predicted IRM Mask)}
\end{array}$$

## 4. Training Stability

We use three key techniques to ensure the model learns effectively:
* MSE Loss : Standard for regression tasks.
* Gradient Clipping : Prevents "exploding gradients" from noisy outliers.
* Learning Rate Scheduler : Slows down training as we approach the minimum to ensure fine-grained weight adjustments.

In [10]:
# Simplified Training Logic
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

# Inside the loop:
# loss.backward()
# torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
# optimizer.step()

## 5. Inference and Reconstruction

During inference, we don't have the clean speech. We take the noisy audio, apply the predicted mask to its magnitude, and keep its original phase for the inverse transformation.

In [11]:
def enhance_example(model, noisy_wav_path, stats):
    # 1. Preprocess: STFT -> Log-Magnitude
    # 2. Windowing: Create context batches
    # 3. Predict: Get masks from MLP
    # 4. Mask: Enhanced_Mag = Noisy_Mag * Pred_Mask
    # 5. Reconstruct: iSTFT(Enhanced_Mag, Noisy_Phase)
    pass